# Radiate Analysis for List of Metabolites
Give a list of Metabolite names, using radiate analysis to find converged reactions, pathways etc.
- 1. Find the chemicals (ChEBI id) using name. Please note that the metabolites used in pathways may not be the free form name provided
- 2. Use ChEBI id to match ReferenceEntity and then find the related Physical Entities
- 3. Peform radiate analysis using Physical Entities

In [1]:
import os
import warnings
from pathlib import PurePosixPath, Path

# Ignore warnings
warnings.filterwarnings('ignore')

## Settings

In [2]:
import dotenv

# Load environment variables from .env file
dotenv.load_dotenv()

# Directory where to look for input data
input_dir = PurePosixPath('input')
print(Path(input_dir).resolve())

# Directory where to output results
output_dir = PurePosixPath('output')
os.makedirs(output_dir, exist_ok=True)

/Users/robin/dev/lifelike-gds/notebooks/reactome/input


In [ ]:
source_file = 'metabolites_chebi.csv'
source_path = input_dir / source_file
print(f"Reading data from {source_path}")


Reading data from input/metabolites_chebi.csv


## Read input file, get list of source ChEBI Ids

In [12]:
import pandas as pd
df = pd.read_csv(source_path)
df['chebi-id'] = df['chebi-id'].astype(str).str.replace('CHEBI:', '')
print(df.head())

chebi_ids = df['chebi-id'].tolist()
print(f"Found {len(chebi_ids)} unique CHEBI IDs in the input data")



  metabolite chebi-id                                     note
0  Vitamin A    12777                             generic term
1  Vitamin A    17336  common pathway form (all-trans-retinol)
2  Vitamin C   176783                             generic term
3  Vitamin C    29073      neutral acid form (L-ascorbic acid)
4  Vitamin C    38290        common pathway form (L-ascorbate)
Found 24 unique CHEBI IDs in the input data


In [7]:
# Matching CHEBI IDs to Reactome nodes, and export the results to an Excel file
from lifelike_gds.neo4j_network.reactome import ReactomeDB
neo4j_db = ReactomeDB()
query = """
match (r:ReferenceMolecule)
where r.databaseName = "ChEBI" and r.identifier in $metabs  
optional match (r)<-[:referenceEntity]-(p:PhysicalEntity)    
return r.stId as chebi, r.name, p.stId, p.compartment, p.synonyms
order by p.compartment
"""
results = neo4j_db.connection.get_dataframe(query, parameters={'metabs': chebi_ids})
print(f"Found {len(results)} matching nodes in Reactome for the CHEBI IDs")

INFO: Successfully connected to Neo4j at bolt://localhost:7687


Found 31 matching nodes in Reactome for the CHEBI IDs


In [14]:
results['chebi'] = results['chebi'].astype(str).str.replace('chebi:', '', regex=False)
merged_df = pd.merge(df, results, left_on='chebi-id', right_on='chebi', how='left')
print(f"After merging with Reactome data, we have {len(merged_df)} rows")
merged_df.to_excel(output_dir/"metabolites_chebi_physicalentity.xlsx", index=False)

After merging with Reactome data, we have 46 rows


In [11]:
# Left-merge the input dataframe with the Reactome results and export to Excel.
results['chebi'] = results['chebi'].astype(str).str.replace('chebi:', '', regex=False)
merged_df = pd.merge(df, results, left_on='chebi-id', right_on='chebi', how='left')
merged_df.to_excel(output_path, index=False)
print(merged_df.head())
print(f"Merged results exported to {output_path}")

         chebi r.name         p.stId p.compartment  \
0  chebi:38290  38290   R-ALL-198816     [cytosol]   
1  chebi:57925  57925    R-ALL-29450     [cytosol]   
2  chebi:17336  17336  R-ALL-2473607     [cytosol]   
3  chebi:17336  17336  R-ALL-2473617     [cytosol]   
4  chebi:17336  17336  R-ALL-2473610     [cytosol]   

                                          p.synonyms metabolite chebi-id  
0                                 [AscH-, Ascorbate]        NaN      NaN  
1  [GSH, Reduced glutathione, glutathione, 5-L-Gl...        NaN      NaN  
2  [atROL, Retinol, all-trans-retinol, Vitamin A,...        NaN      NaN  
3  [atROL, Retinol, all-trans-retinol, Vitamin A,...        NaN      NaN  
4  [atROL, Retinol, all-trans-retinol, Vitamin A,...        NaN      NaN  


## Find Physical Entities based on ChEBI Ids

In [8]:
from lifelike_gds.arango_network.reactome import *
database = ReactomeDB(os.getenv('ARANGO_DATABASE', 'reactome'),
                      os.getenv('ARANGO_URI', 'localhost'), 
                      os.getenv('ARANGO_USER', 'root'), 
                      os.getenv('ARANGO_PASSWORD', ''))

In [16]:
nodes = database.get_entity_nodes_by_chebi_ids(chebi_ids)
print(f"Found {len(nodes)} nodes in Reactome matching the CHEBI IDs")


INFO: 24 chebi_ids, matched to 31 nodes


Found 31 nodes in Reactome matching the CHEBI IDs


## Load graph from arango graph database to memery

In [18]:
from lifelike_gds.arango_network.reactome import Reactome
from lifelike_gds.arango_network.radiate_trace import RadiateTrace

tracegraph = RadiateTrace(Reactome(database))

# set up output directory where the excel and graph files will write to
tracegraph.datadir = output_dir

# initiate tracegraph by loading graph data from arango
# a networkx graph is created here.
tracegraph.init_default_graph()

INFO: load reactome graph
INFO: MultiDirectedGraph with 71225 nodes and 112575 edges


In [19]:
"""
Perform radiate analysis from the given source_nodes, and export pageranks and rev_pageranks
into excel file. The excel file contains two tabs, one for pageranks and one for reverse pageranks.
The data are sorted by pagerank (or rev_pagerank)
rows_export: define the top ranked rows exported into file
"""


def export_radiate_analysis(
    tracegraph,
    source_name,
    source_decription,
    source_nodes,
    exclude_sources_from_file=False,
    rows_export=4000,
):
    tracegraph.graph = tracegraph.orig_graph.copy()
    tracegraph.set_node_set_from_arango_nodes(
        source_nodes, source_name, source_decription
    )
    outfile_name = f"Radiate_analysis_for_{source_name}.xlsx"
    tracegraph.export_pagerank_data(
        source_name,
        outfile_name,
        direction='both',
        num_nodes=rows_export,
        exclude_sources=exclude_sources_from_file,
    )

In [20]:
source_name = 'metabolites'
source_desc = 'abnormal metabolites from patient diagnosis'
export_radiate_analysis(
    tracegraph,
    source_name,
    source_desc,
    nodes
)

INFO: set pagerank and num reach for metabolites
INFO: export top 4000 pagerank data into output/Radiate_analysis_for_metabolites.xlsx
